<a href="https://colab.research.google.com/github/safeai-snu/finance_ai_2025/blob/main/day4/TS_forecasting_gold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q neuralforecast utilsforecast
!pip show neuralforecast

from neuralforecast import NeuralForecast
from neuralforecast.models import MLP
from neuralforecast.losses.pytorch import MAE
from neuralforecast.tsdataset import TimeSeriesDataset

import pytorch_lightning as pl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 257.7/257.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.8/285.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.1/823.1 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

### Metal price prediction (gold, silver, platinum)

In [3]:
gold_url = 'https://raw.githubusercontent.com/SLCFLAB/DL-Forecasting/main/DL/day1/data/gold.csv'
#silver_url = 'https://raw.githubusercontent.com/SLCFLAB/DL-Forecasting/main/DL/day1/data/silver.csv'
#platinum_url = 'https://raw.githubusercontent.com/SLCFLAB/DL-Forecasting/main/DL/day1/data/platinum.csv'

gold = pd.read_csv(gold_url, index_col=0, parse_dates=['Date'])
#silver = pd.read_csv(silver_url, index_col=0, parse_dates=['Date'])
#platinum = pd.read_csv(platinum_url, index_col=0, parse_dates=['Date'])

In [4]:
def preprocess(df, name):
    #df = df.reset_index()  # Date → 열로
    df = df.rename(columns={'Date': 'ds', df.columns[1]: 'y'})  # 두 번째 열이 가격이라고 가정
    df['unique_id'] = name
    #df['trend'] = df['y'].expanding().mean()  # 예시 trend: 누적 평균
    #df['y_[lag12]'] = df['y'].shift(12)       # 12개월 전 값
    #df['month'] = df['ds'].dt.month
    return df[['unique_id', 'ds', 'y']]

def temporal_split(df, h=12):
    train_list = []
    test_list = []

    for uid, gdf in df.groupby('unique_id'):
        gdf_sorted = gdf.sort_values('ds').reset_index(drop=True)
        train = gdf_sorted.iloc[:-h]
        test = gdf_sorted.iloc[-h:]
        train_list.append(train)
        test_list.append(test)

    Y_train_df = pd.concat(train_list).reset_index(drop=True)
    Y_test_df = pd.concat(test_list).reset_index(drop=True)
    return Y_train_df, Y_test_df

In [5]:
gold_df = preprocess(gold, 'gold')
#silver_df = preprocess(silver, 'silver')
#platinum_df = preprocess(platinum, 'platinum')

#combined_df = pd.concat([gold_df, silver_df, platinum_df], axis=0).sort_values(by=['unique_id', 'ds']).reset_index(drop=True)

horizon = 20
Y_train_df, Y_test_df = temporal_split(gold_df, h=horizon)

### Training Forecasting Models

In [ ]:
from neuralforecast.models import NBEATS, NHITS, PatchTST
#from neuralforecast.models import DLinear, Autoforemr, Informer, DeepAR, iTransformer

models = [
    NBEATS(h=horizon, input_size=4*horizon, loss=MAE(), max_steps=300),
    NHITS(h=horizon, input_size=4*horizon, loss=MAE(), max_steps=300),
    PatchTST(h=horizon, input_size=4*horizon, loss=MAE(), max_steps=300, patch_len=5, stride=5),
    # PatchTST(h=horizon,
    #         input_size=2*horizon,
    #         patch_len=25,
    #         stride=5,
    #         revin=False,
    #         hidden_size=16,
    #         n_heads=4,
    #         scaler_type='robust',
    #         loss=MAE(),
    #         learning_rate=1e-3,
    #         max_steps=300,
    #         val_check_steps=50,
    #         early_stop_patience_steps=2)
]

nf = NeuralForecast(models=models, freq='B')
nf.fit(df=Y_train_df, val_size=horizon)

INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.6 M  | train
-------------------------------------------------------
2.6 M     Trainable params
4.1 K     Non-trainable params
2.6 M     Total params
10.343    Total estimated model params size (MB)
31        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.6 M  | train
-------------------------------------------------------
2.6 M     Trainable params
0         Non-trainable params
2.6 M     Total params
10.354    Total estimated model params size (MB)
34        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [ ]:
missing = nf.get_missing_future(Y_test_df)
missing['y'] = np.nan
futr_df = pd.concat([Y_test_df, missing], axis=0).sort_values(['unique_id', 'ds']).reset_index(drop=True)

forecasts = nf.predict(futr_df=futr_df)

In [ ]:
from utilsforecast.plotting import plot_series
plot_series(gold_df[-horizon*8:], forecasts)

In [ ]:
from sklearn.metrics import mean_absolute_error

mae_scores = {}

for uid in forecasts['unique_id'].unique():
    mae_scores[uid] = {}
    y_true = Y_test_df[Y_test_df['unique_id'] == uid]['y'].values

    for col in forecasts.columns:
        if col in ['ds', 'unique_id'] or col.endswith('lo-95') or col.endswith('hi-95'):
            continue

        y_pred = forecasts[forecasts['unique_id'] == uid][col].values
        y_pred = y_pred[:len(y_true)]  # 길이 맞추기 (예방적)
        mae = mean_absolute_error(y_true, y_pred)
        mae_scores[uid][col] = mae

print("MAE Scores (by series and model):")
for uid, model_dict in mae_scores.items():
    print(f"\n{uid}:")
    for model, score in model_dict.items():
        print(f"  {model}: {score:.3f}")

In [ ]:

Y_hat_df = forecasts.reset_index(drop=True).drop(columns=['unique_id','ds'])
plot_df_nf = pd.concat([Y_test_df.reset_index(drop=True), Y_hat_df], axis=1)
plot_df = pd.concat([Y_train_df, plot_df_nf])

plot_df = plot_df[plot_df['unique_id'] == 'gold']

plt.figure(figsize=(12, 6))
plt.plot(plot_df['ds'], plot_df['y'], label='True', c='grey')

for col in forecasts.columns:
    if 'median' in col or col.startswith('ds') or col.startswith('unique_id'):
        continue
    if 'PatchTST' in col and '-lo-' in col:
        continue  # Skip low/high intervals for now
    plt.plot(plot_df['ds'], plot_df[col], label=col)

plt.legend()
plt.title("Forecast Comparison")
plt.grid()
plt.tight_layout()
plt.show()

## StatsForecast

In [ ]:
!pip install statsforecast

In [ ]:
import matplotlib.pyplot as plt

dark_style = {
    'figure.facecolor': '#212946',
    'axes.facecolor': '#212946',
    'savefig.facecolor': '#212946',
    'axes.grid': True,
    'axes.grid.which': 'both',
    'grid.color': '#2A3459',
    'grid.linewidth': 1,
    'axes.spines.left': False,
    'axes.spines.right': False,
    'axes.spines.top': False,
    'axes.spines.bottom': False,
    'text.color': '#FFFFFF',
    'axes.labelcolor': '#FFFFFF',
    'xtick.color': '#FFFFFF',
    'ytick.color': '#FFFFFF',
    'font.size': 12,
    'lines.linewidth': 2
}
plt.rcParams.update(dark_style)

from pylab import rcParams
rcParams['figure.figsize'] = (18,7)

import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf

## Time Series Analysis

In [ ]:
df_p = gold_df

In [ ]:
from statsforecast import StatsForecast

StatsForecast.plot(df_p)

In [ ]:
df = df_p[df_p['unique_id']=='gold']
df["y"].plot(kind='kde',figsize = (16,5))
df["y"].describe()

In [ ]:
# Autocorrelation plot
fig, axs = plt.subplots(nrows=1, ncols=2)

plot_acf(df["y"],  lags=60, ax=axs[0],color="fuchsia")
axs[0].set_title("Autocorrelation");

plot_pacf(df["y"],  lags=60, ax=axs[1],color="lime")
axs[1].set_title('Partial Autocorrelation')

plt.show();

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
decomposed=seasonal_decompose(df["y"], model = "add", period=20)
decomposed.plot()
plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller

In [ ]:
def Augmented_Dickey_Fuller_Test_func(series , column_name):
    print (f'Dickey-Fuller test results for columns: {column_name}')
    dftest = adfuller(series, autolag='AIC')
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','No Lags Used','Number of observations used'])
    for key,value in dftest[4].items():
        dfoutput['Critical Value (%s)'%key] = value
    print (dfoutput)
    if dftest[1] <= 0.05:
        print("Conclusion:====>")
        print("Reject the null hypothesis")
        print("The data is stationary")
    else:
        print("Conclusion:====>")
        print("The null hypothesis cannot be rejected")
        print("The data is not stationary")

In [ ]:
Augmented_Dickey_Fuller_Test_func(df["y"],"metal")

In [ ]:
df1=df.copy()
df1['y_diff'] = df['y'].diff()
df1.dropna(inplace=True)
df1.head()

In [ ]:
Augmented_Dickey_Fuller_Test_func(df1["y_diff"],"metal_diff")

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt


fig, axes = plt.subplots(2, 2, )
axes[0, 0].plot(df1["y"]); axes[0, 0].set_title('Original Series')
plot_acf(df1["y"], ax=axes[0, 1],lags=20)

axes[1, 0].plot(df1["y"].diff()); axes[1, 0].set_title('1st Order Differencing')
plot_acf(df1["y"].diff().dropna(), ax=axes[1, 1],lags=20)


plt.show()

##  AutoARIMA, AutoETS, AutoTheta with StatsForecast

In [ ]:
df["ds"] = pd.to_datetime(df["ds"])

Y_train_df.shape, Y_test_df.shape

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, AutoTheta
from statsforecast.arima import arima_string

In [ ]:
season_length = horizon
horizon = len(Y_test_df)

models = [AutoARIMA(season_length=season_length),
          AutoETS(model='ZZZ', season_length=season_length),
          AutoTheta(season_length=season_length,
                     decomposition_type="additive",
                     model="STM")]

In [ ]:
sf = StatsForecast(models=models, freq='B')
sf.fit(df=Y_train_df)

#### Residual analysis

In [ ]:
arima_string(sf.fitted_[0,0].model_)

In [ ]:
#series index, model index
result=sf.fitted_[0,0].model_
print(result.keys())
print(result['arma'])

In [ ]:
#series index, model index
result=sf.fitted_[0,2].model_
print(result.keys())

In [ ]:
residual=pd.DataFrame(result.get("residuals"), columns=["residual Model"])
residual

In [ ]:
from scipy import stats


fig, axs = plt.subplots(nrows=2, ncols=2)

# plot[1,1]
residual.plot(ax=axs[0,0])
axs[0,0].set_title("Residuals");

# plot
sns.distplot(residual, ax=axs[0,1]);
axs[0,1].set_title("Density plot - Residual");

# plot
stats.probplot(residual["residual Model"], dist="norm", plot=axs[1,0])
axs[1,0].set_title('Plot Q-Q')

# plot
plot_acf(residual,  lags=35, ax=axs[1,1],color="fuchsia")
axs[1,1].set_title("Autocorrelation");

plt.show();

### Forecasting results

In [ ]:
Y_train_df = Y_train_df[['unique_id', 'ds', 'y']]
Y_test_df = Y_test_df[['unique_id', 'ds', 'y']]

In [ ]:
Y_hat_df = sf.forecast(df=Y_train_df, h=horizon, fitted=True)

In [ ]:
Y_hat_df

In [ ]:
sf.plot(Y_train_df, Y_hat_df)

In [ ]:
# values=sf.forecast_fitted_values()
# values

In [ ]:
forecasts = sf.forecast(df=Y_train_df, h=horizon, level=[95])

uid = 'gold'
fcst = forecasts[forecasts['unique_id'] == uid]

# 실제값
y_train = Y_train_df[Y_train_df['unique_id'] == uid]
y_test = Y_test_df[Y_test_df['unique_id'] == uid]

plt.figure(figsize=(12, 6))

plt.plot(y_train['ds'], y_train['y'], label='Train', color='white')
plt.plot(y_test['ds'], y_test['y'], label='Test', color='gray')

# 예측값 (예: AutoARIMA)
plt.plot(fcst['ds'], fcst['AutoARIMA'], label='Forecast', color='#00C4FF')

plt.fill_between(fcst['ds'],
                 fcst['AutoARIMA-lo-95'],
                 fcst['AutoARIMA-hi-95'],
                 color='#00C4FF', alpha=0.2, label='95% CI')

plt.title(f'Forecast for {uid}', color='white')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plot_df_sf = Y_test_df.merge(Y_hat_df, how='left', on=['unique_id', 'ds'])
plot_df_sf

In [ ]:
model_names = ['AutoARIMA', 'AutoETS', 'AutoTheta', 'NBEATS', 'NHITS', 'PatchTST', ]
cols = ['unique_id', 'ds', 'y'] +  model_names
fcst = pd.concat([plot_df_nf, plot_df_sf.drop(columns=['unique_id', 'ds', 'y'])], axis=1)[cols]

In [ ]:
sf.plot(Y_train_df, fcst)
#sf.plot(Y_train_df, fcst, unique_ids=['Airline1'])

In [ ]:
from functools import partial

import utilsforecast.losses as ufl
from utilsforecast.evaluation import evaluate

In [ ]:
evals = evaluate(
    fcst,
    metrics=[ufl.mae, ufl.mape, partial(ufl.mase, seasonality=season_length), ufl.rmse, ufl.smape],
    train_df=Y_train_df,
).sort_values(by='unique_id')

evals

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

metrics = ['mae', 'mape', 'mase', 'rmse', 'smape']

uid = 'gold'
df_series = evals[evals['unique_id'] == uid].set_index('metric').loc[metrics, model_names]

# row-wise Z-score 정규화
df_scaled = pd.DataFrame(index=metrics, columns=model_names, dtype=float)
for metric in metrics:
    row = df_series.loc[metric]
    if row.isna().all():
        continue
    scaled_row = StandardScaler().fit_transform(row.dropna().values.reshape(-1, 1)).flatten()
    df_scaled.loc[metric, row.dropna().index] = scaled_row

angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]

# plot
fig, ax = plt.subplots(figsize=(7, 6), subplot_kw=dict(polar=True))

for model in model_names:
    if df_scaled[model].isna().any():
        continue
    values = df_scaled[model].tolist()
    values += values[:1]
    ax.plot(angles, values, label=model, linewidth=2)
    ax.fill(angles, values, alpha=0.1)

ax.set_thetagrids(np.degrees(angles[:-1]), metrics)
ax.set_title(f'Model Performance (Z-Score): {uid}', y=1.08)
ax.grid(True)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.show()